In [ ]:
import sys, os

# Get the directory where the notebook is running
notebook_dir = os.getcwd()

# Go up three levels: models → src → backend → project root
project_root = os.path.abspath(
    os.path.join(notebook_dir, "..", "..")
)

# Add project root to Python path
if project_root not in sys.path:
    sys.path.append(project_root)

print("Project root:", project_root)
print("src in path:", any("src" in p for p in sys.path))

In [ ]:
# Connect to Qdrant from the Windows-hosted notebook
from qdrant_client import QdrantClient, models

qdrant_url = "http://localhost:6333"
qdrant = QdrantClient(url=qdrant_url, check_compatibility=False)
print(f"Connected to Qdrant at {qdrant_url}")

In [ ]:
#List collections
qdrant.get_collections()

In [ ]:
#Check if a collection exists
qdrant.collection_exists("doc_intelligence")

In [ ]:
# Generate embeddings
import src.services.rag.retrieval_engine as retrieval_engine
from src.services.rag.retrieval_engine import queryRetrieval
from src.services.model_dependencies.session_maker import async_session_maker

# queryRetrieval normally creates a Docker-network client using qdrant:6333.
# Replace it with the Windows-hosted notebook client.
retrieval_engine.qdrant = qdrant

text = "how many blow wholes does a baleen whale have"
async with async_session_maker() as session:
    doc = queryRetrieval(session=session)
    embedding = doc.hybrid_search(text)

print(embedding)

In [ ]:
import src.services.rag.retrieval_engine as retrieval_engine
from src.services.rag.retrieval_engine import queryRetrieval
from src.services.model_dependencies.session_maker import async_session_maker

retrieval_engine.qdrant = qdrant

async with async_session_maker() as session:
    doc = queryRetrieval(session=session)
    response = doc.answer(
        id='ebe526d5-320f-45dd-9290-5569efe1c231',
        query="where do baleen whales live",
        username="alex_morgan",
        rewrite_model="qwen2.5:3b",
        generation_model="qwen3.8:27b",
        provider="ollama",
    )
    async for chunk in response:
        print(chunk)

In [ ]:
from src.services.model_dependencies.database import engine
print(engine.url.host, engine.url.port)

In [ ]:
from sqlalchemy.ext.asyncio import async_sessionmaker, create_async_engine
from src.services.model_dependencies.database import engine

notebook_engine = create_async_engine(
    engine.url.set(host="localhost", port=5432)
)
notebook_session_maker = async_sessionmaker(
    notebook_engine,
    expire_on_commit=False,
)

In [ ]:
import pandas as pd
from sqlalchemy import select
from src.models.user_querydb import Conversation, ChatMessage, MessageContext

async def fetch_conversation_history(conversation_id: str):
    async with notebook_session_maker() as session:
        stmt = (
            select(ChatMessage, Conversation, MessageContext)
            .join(Conversation, ChatMessage.conversation_id == Conversation.id)
            .join(MessageContext, MessageContext.message_id == ChatMessage.id)
            .where(Conversation.id == conversation_id)
            .order_by(ChatMessage.created_at.asc())
        )

        result = await session.execute(stmt)

        rows = []
        for msg, conv, ctx in result.all():
            rows.append({
                "conversation_id": str(conv.id),
                "username": conv.username,
                "conversation_created": conv.created_at,
                "conversation_updated": conv.last_updated,
                "message_id": msg.id,
                "query": msg.query,
                "response": msg.response,
                "rephrase": msg.rephrase,
                "message_created": msg.created_at,
                "context": ctx.context,
            })

        return pd.DataFrame(rows)


conversation_id = "d79ca566-c5b8-4efd-9f4a-cc2d8a81e709"
table = await fetch_conversation_history(conversation_id)
table
